In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             accuracy_score, precision_recall_curve)
from imblearn.over_sampling import SMOTE

# ── 1. Load & encode ──────────────────────────────────────────────
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df = df.drop(columns=["id"], errors="ignore")
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df = pd.get_dummies(df, drop_first=True)

X = df.drop(columns=["stroke"])
y = df["stroke"]

# ── 2. Split FIRST — before anything else ────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # keeps 5% stroke ratio in both splits
)

print("Train class dist:", y_train.value_counts().to_dict())
print("Test  class dist:", y_test.value_counts().to_dict())

# ── 3. Scale ONLY on training data ───────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)   # fit+transform train
X_test_scaled  = scaler.transform(X_test_raw)        # transform only test

# ── 4. Apply SMOTE ONLY on scaled training data ──────────────────
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_scaled, y_train)

print("After SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())

# ── 5. Train Logistic Regression with Grid Search ─────────────────
lr = LogisticRegression(max_iter=1000, solver='liblinear')

param_grid = {
    'C': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
    'penalty': ['l1', 'l2'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    lr, param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_bal, y_train_bal)

best_lr = grid.best_estimator_
print("\nBest params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

# ── 6. Evaluate on untouched test set ────────────────────────────
y_prob = best_lr.predict_proba(X_test_scaled)[:, 1]
y_pred = best_lr.predict(X_test_scaled)

print("\n=== Default threshold (0.5) ===")
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC :", roc_auc_score(y_test, y_prob))
print("Accuracy:", accuracy_score(y_test, y_pred))

# ── 7. Threshold scan ─────────────────────────────────────────────
print(f"\n{'Threshold':>10} {'Accuracy':>10} {'Recall':>8} {'Precision':>10} {'F1':>6}")
print("-" * 50)
for t in np.arange(0.10, 0.65, 0.05):
    p = (y_prob >= t).astype(int)
    rep = classification_report(y_test, p, output_dict=True, zero_division=0)
    print(f"{t:>10.2f} "
          f"{accuracy_score(y_test,p):>10.3f} "
          f"{rep['1']['recall']:>8.3f} "
          f"{rep['1']['precision']:>10.3f} "
          f"{rep['1']['f1-score']:>6.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'healthcare-dataset-stroke-data.csv'